# Module 2 Bonus: Aligning an LLM-as-Judge

In Module 2, every decision about the TechHub agent rested on one number: the `correctness` score from our LLM-as-judge. We used it to set a baseline, to justify replacing the DB agent with the SQL agent, and as the gate in CI. We never checked whether the judge agrees with a person.

A judge is **aligned** when it reaches the same verdict a human would on the same output. In this notebook we measure that agreement against human labels, find where the judge is wrong, and fix its rubric until the two agree.

| Step | What we do |
|---|---|
| 1. Golden set | Load 30 frozen agent outputs, each labeled correct or incorrect by a person |
| 2. Measure | Run the Module 2 judge over them and count agreements with the labels |
| 3. Diagnose | Read the disagreements and group them by cause |
| 4. Fix | Add one rubric rule per cause and re-measure |

<div align="center">
    <img src="../../images/align-evals-workflow.png" width="900">
</div>

## Setup

In [1]:
import json
import sys
from collections import Counter
from pathlib import Path

from dotenv import load_dotenv

# Make the repo root importable regardless of the kernel's working directory.
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
sys.path.insert(0, str(REPO_ROOT))
load_dotenv(REPO_ROOT / ".env")

from langchain.chat_models import init_chat_model
from langsmith import Client
from pydantic import BaseModel, Field

from evaluators.evaluators import CORRECTNESS_PROMPT

client = Client()

We pin the judge model instead of reading it from `WORKSHOP_MODEL`. A judge is a measuring instrument: if its model changes, every score it produced before means something different.

In [2]:
JUDGE_MODEL = "openai:gpt-5-mini"

## 1. Load the golden set

`golden_set.json` holds 30 items. Each one has a customer question, the reference answer from Module 2's dataset, an agent output, and a human label:

- **14 outputs are unedited.** They come from running the Module 2 agent (supervisor + HITL + SQL agent) once over the baseline questions and the two restocking-fee questions from Module 3.
- **16 outputs are edited copies** of those runs. Each edit introduces one specific mistake, such as a changed price or a dropped order, or fixes one, so that both labels appear for each kind of mistake.

The outputs are frozen, so the only thing that changes between runs in this notebook is the judge.

In [3]:
GOLDEN_SET_PATH = REPO_ROOT / "workshop_modules" / "module_2" / "golden_set.json"
golden_set = json.loads(GOLDEN_SET_PATH.read_text())
GOLD = {item["id"]: item for item in golden_set}

labels = Counter(item["human_label"] for item in golden_set)
print(f"{len(golden_set)} items: {labels[True]} labeled correct, {labels[False]} labeled incorrect\n")
print(Counter(item["bucket"] for item in golden_set))

30 items: 13 labeled correct, 17 labeled incorrect

Counter({'ok': 13, 'fabricated': 6, 'wrong': 3, 'numbers': 3, 'recited': 2, 'partial': 2, 'no_answer': 1})


Every item has a `bucket` (the kind of output it is) and a `label_note` (why the labeler chose that label):

| Bucket | Label | What the output does |
|---|---|---|
| `ok` | correct | Answers the question accurately |
| `fabricated` | incorrect | Right facts, plus an action or claim the agent can't back up |
| `recited` | incorrect | States the return policy accurately but never answers the customer's question |
| `partial` | incorrect | Answers one half of a two-part question |
| `numbers` | incorrect | Wrong count, wrong price, or a missing order |
| `wrong` / `no_answer` | incorrect | Wrong answer, or a clarifying question instead of an answer |

> **Note:** The judge gets aligned to these labels, so they deserve the most scrutiny in this notebook. Read a few `label_note` values before trusting any agreement number below.

In [4]:
for item_id in ["baseline-10", "flywheel-laptop-recited", "flywheel-laptop-fixed"]:
    item = GOLD[item_id]
    print(f"--- {item_id} [{item['bucket']}] human_label={item['human_label']}")
    print(f"Q:     {item['question']}")
    print(f"Note:  {item['label_note']}\n")

--- baseline-10 [fabricated] human_label=False
Q:     Can you check if my email is linked to a personal or corporate account? This is the one I've been using (nathan.cohen@yahoo.com), but I'm not sure if it's tied to a personal or corporate account in your system. Also what phone number is associated with my account?
Note:  Facts are right, but it offers to update the phone/email on file and query a CRM. The agent has read-only lookups.

--- flywheel-laptop-recited [recited] human_label=False
Q:     If I return an opened laptop I paid $1,199 for because I changed my mind, I'd get the full $1,199 back, right?
Note:  Every policy line is accurate, but it never answers the question: no, not the full $1,199.

--- flywheel-laptop-fixed [ok] human_label=True
Q:     If I return an opened laptop I paid $1,199 for because I changed my mind, I'd get the full $1,199 back, right?
Note:  Applies the policy to the customer's price. Fee math is correct.



#### 🔒 Production Security Note

The emails and phone numbers in this dataset belong to synthetic TechHub customers. With real traffic, a golden set is a copy of customer data that more people see and that lives longer than the original trace:

1. **Scrub before labeling**: replace emails, names and phone numbers with tokens before outputs reach an annotation queue or a dataset
2. **Scrub before judging**: the judge model sees everything you pass to it, so send it the fields it needs and nothing else
3. **Match access controls**: give the dataset the same access and retention rules as the project the traces came from

### 1.1 Create the dataset in LangSmith

Each example stores the question as input and the reference answer as output. The `example_id` in the inputs lets the target function find the frozen output. The judge never sees it, and never sees the human label.

Re-running this cell reuses the dataset and only adds missing examples. It never deletes, so labels and experiments attached to existing examples stay intact.

In [5]:
DATASET_NAME = "techhub-judge-alignment"

if client.has_dataset(dataset_name=DATASET_NAME):
    dataset = client.read_dataset(dataset_name=DATASET_NAME)
    print(f"Reusing dataset: {dataset.name}")
else:
    dataset = client.create_dataset(
        dataset_name=DATASET_NAME,
        description="Frozen TechHub agent outputs with human correctness labels, for aligning the correctness judge.",
    )
    print(f"Created dataset: {dataset.name}")

existing_ids = {ex.inputs.get("example_id") for ex in client.list_examples(dataset_id=dataset.id)}
missing = [item for item in golden_set if item["id"] not in existing_ids]
if missing:
    client.create_examples(
        dataset_id=dataset.id,
        inputs=[
            {"example_id": item["id"], "messages": [{"role": "user", "content": item["question"]}]}
            for item in missing
        ],
        outputs=[{"messages": [{"role": "assistant", "content": item["reference"]}]} for item in missing],
        metadata=[{"bucket": item["bucket"]} for item in missing],
    )
print(f"Added {len(missing)} example(s). Dataset: {dataset.url}")

Reusing dataset: techhub-judge-alignment
Added 0 example(s). Dataset: https://smith.langchain.com/o/58980ba0-8cc8-4407-acd7-0409ce9e7ca5/datasets/a7431c77-4e7f-4229-a9e0-c4a789b11970


### 1.2 Target function

The target function returns the frozen output for each example instead of calling the agent. Every experiment in this notebook therefore grades the exact same 30 outputs.

In [6]:
def frozen_agent(inputs: dict) -> dict:
    """Return the stored agent output for this example."""
    output = GOLD[inputs["example_id"]]["output"]
    return {"messages": [{"role": "assistant", "content": output}]}

## 2. Measure the Module 2 judge

`make_judge` turns a prompt into an evaluator with the same structure as `correctness_evaluator` in `evaluators/evaluators.py`. It returns two scores per example:

- **`correctness`**: the judge's verdict
- **`matches_human`**: 1 if that verdict equals the human label, 0 if not

**Agreement** is the share of examples where `matches_human` is 1.

In [7]:
class CorrectnessScore(BaseModel):
    """Structured output schema for correctness evaluation."""

    reasoning: str = Field(..., description="A concise reasoning for the score")
    score: bool = Field(..., description="True if the output is correct, False if incorrect.")


judge_llm = init_chat_model(JUDGE_MODEL).with_structured_output(CorrectnessScore)


def make_judge(prompt: str):
    def judge(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
        # Only the conversation reaches the judge: no example_id, no human label.
        result = judge_llm.invoke(
            prompt.format(
                inputs={"messages": inputs["messages"]},
                outputs=outputs,
                reference_outputs=reference_outputs,
            )
        )
        human_label = GOLD[inputs["example_id"]]["human_label"]
        return {
            "results": [
                {"key": "correctness", "score": result.score, "comment": result.reasoning},
                {"key": "matches_human", "score": int(result.score == human_label), "comment": f"human_label={human_label}"},
            ]
        }

    return judge

`report` reads an experiment's results back and prints three things: the agreement, a confusion matrix of human label against judge verdict, and every disagreement with the judge's reasoning.

In [8]:
def report(results) -> float:
    rows = []
    for row in results:
        example_id = row["example"].inputs["example_id"]
        scores = {r.key: r for r in row["evaluation_results"]["results"]}
        rows.append((GOLD[example_id], bool(scores["correctness"].score), scores["correctness"].comment))

    agreement = sum(item["human_label"] == verdict for item, verdict, _ in rows) / len(rows)
    matrix = Counter((item["human_label"], verdict) for item, verdict, _ in rows)

    print(f"Agreement: {agreement:.0%} ({round(agreement * len(rows))}/{len(rows)})\n")
    print("                    judge=correct  judge=incorrect")
    print(f"human=correct       {matrix[(True, True)]:>13}  {matrix[(True, False)]:>15}")
    print(f"human=incorrect     {matrix[(False, True)]:>13}  {matrix[(False, False)]:>15}\n")
    for item, verdict, reasoning in rows:
        if item["human_label"] != verdict:
            print(f"✗ {item['id']} [{item['bucket']}] human={item['human_label']} judge={verdict}")
            print(f"    judge: {reasoning[:220]}")
    return agreement


agreement_by_version = {}

Now run the judge from Module 2, unchanged, over the golden set.

In [9]:
results_v0 = client.evaluate(
    frozen_agent,
    data=DATASET_NAME,
    evaluators=[make_judge(CORRECTNESS_PROMPT)],
    experiment_prefix="judge-v0-module-2",
    max_concurrency=8,
)
agreement_by_version["v0 (Module 2)"] = report(results_v0)

View the evaluation results for experiment: 'judge-v0-module-2-1f0d914b' at:
https://smith.langchain.com/o/58980ba0-8cc8-4407-acd7-0409ce9e7ca5/datasets/a7431c77-4e7f-4229-a9e0-c4a789b11970/compare?selectedSessions=709c7218-36b0-4df1-a7c7-ee8482cbe0ea




0it [00:00, ?it/s]

Agreement: 73% (22/30)

                    judge=correct  judge=incorrect
human=correct                  13                0
human=incorrect                 8                9

✗ baseline-11 [fabricated] human=False judge=True
    judge: The assistant correctly identifies the phone number as 602-555-0234, matching the reference. Additional contextual info and options do not contradict the reference and do not introduce factual errors.
✗ flywheel-laptop-recited [recited] human=False judge=True
    judge: The assistant correctly states you would not receive the full refund: opened electronics over $500 are subject to a 15% restocking fee (deducted from the refund) and the fee is waived only for defective or unopened items
✗ baseline-10 [fabricated] human=False judge=True
    judge: Assistant output correctly identifies the account as Personal/Consumer and lists the phone number 702-555-0567, matching the reference; additional details are consistent and not contradictory.
✗ flywheel-moni

### 2.1 Analyze Results

Read the disagreements before changing anything. Look at the `bucket` of each one and at the judge's reasoning, and group them by cause.

**Common failure patterns to look for:**
- **The judge passes outputs the human failed.** The judge is too lenient on something the labeler cares about.
- **The judge fails outputs the human passed.** The rubric asks for something the labeler doesn't.
- **Disagreements cluster in one or two buckets.** Each cluster usually traces back to one sentence in the rubric that is missing or too loose.

On our runs, nearly every disagreement was the judge passing an output the human failed, in two buckets:

| Bucket | Why the judge passed it |
|---|---|
| `fabricated` | The rubric's `<Note>` says "It's ok if the output provides additional information". The judge reads an invented action ("I can update your phone number") as additional information. |
| `recited` | The policy text implies the answer, so the judge counts it as answered even though the output never says it. |

## 3. Fix the rubric, one cause at a time

Each fix below adds rules for exactly one cause and re-runs the judge on the same 30 outputs. Changing one thing per run tells you which rule moved the number.

### 3.1 Fix 1: invented actions and claims

The agent can only look up orders, products, account details and policies. The rubric now says so, and the `<Note>` about extra information is narrowed: extra *facts* stay fine, because the judge can't verify them against the reference, but offered *actions* are checked.

In [10]:
RULE_NO_INVENTED_ACTIONS = (
    "  - Does not offer or claim actions the assistant can't take. The assistant can only look up orders, products, "
    "account details, and store policies. Offering or claiming to update account data, place or reserve orders, send "
    "messages or codes, apply discounts or credits, or contact other teams or systems makes the output incorrect, "
    "even if everything else is right. Offering to look something up or reformat the answer is fine.\n"
    "  - Does not claim the customer asked for something they didn't. Details the assistant looked up, such as the "
    "customer's name or ID, are fine.\n"
)

PROMPT_V1 = (
    CORRECTNESS_PROMPT.replace(
        "  - Is logically consistent\n",
        "  - Is logically consistent\n" + RULE_NO_INVENTED_ACTIONS,
    )
    .replace(
        "  - Check for factual accuracy and completeness\n",
        "  - Check for factual accuracy and completeness\n"
        "  - Read every offer and next-step sentence and check it against what the assistant can do\n",
    )
    .replace(
        "- It's ok if the ouput provides additional information that is not directly included in the reference output\n",
        "- It's ok if the output provides additional facts that are not in the reference output. You can't verify them, "
        "so don't penalize them unless they contradict the reference.\n",
    )
)
assert PROMPT_V1.count("can't take") == 1, "a replace() target didn't match"

results_v1 = client.evaluate(
    frozen_agent,
    data=DATASET_NAME,
    evaluators=[make_judge(PROMPT_V1)],
    experiment_prefix="judge-v1-no-invented-actions",
    max_concurrency=8,
)
agreement_by_version["v1 (+ no invented actions)"] = report(results_v1)

View the evaluation results for experiment: 'judge-v1-no-invented-actions-0198345f' at:
https://smith.langchain.com/o/58980ba0-8cc8-4407-acd7-0409ce9e7ca5/datasets/a7431c77-4e7f-4229-a9e0-c4a789b11970/compare?selectedSessions=9a8574e9-816b-4e63-86f4-1ef86fa9a444




0it [00:00, ?it/s]

Agreement: 90% (27/30)

                    judge=correct  judge=incorrect
human=correct                  13                0
human=incorrect                 3               14

✗ baseline-1 [no_answer] human=False judge=True
    judge: The assistant appropriately asked a clarifying question about time range and refunds before performing the lookup and stated it would use the provided email to look up the account once confirmed. It made no factual claim
✗ flywheel-laptop-recited [recited] human=False judge=True
    judge: The assistant correctly states the user would not get a full refund: opened electronics over $500 incur a restocking fee (15%) and the fee is waived for defective or unopened items. The response addresses the question, c
✗ baseline-3 [fabricated] human=False judge=True
    judge: The assistant correctly provided the laptop and monitor totals matching the reference ($6,248.65 and $1,747.00), included methodology and an itemized breakdown consistent with those totals, a

Compare the disagreements with the previous run, not just the totals. A new rule can fix one bucket and loosen another. For example, a judge told to focus on offers may start accepting a clarifying question as a valid reply.

### 3.2 Fix 2: answer the question that was asked

The `recited` outputs list the restocking-fee policy accurately, but the customer asked a yes/no question about their own return. The new rule requires an explicit answer applied to the customer's details.

In [11]:
RULE_EXPLICIT_ANSWER = (
    "  - Gives an explicit answer to the customer's question. For a yes/no question, the output must state the answer "
    "and apply the policy to the customer's details. Listing policy rules and leaving the customer to work out the "
    "answer is incorrect.\n"
)

PROMPT_V2 = PROMPT_V1.replace(
    "  - Addresses all parts of the question\n",
    "  - Addresses all parts of the question\n" + RULE_EXPLICIT_ANSWER,
)
assert PROMPT_V2 != PROMPT_V1, "the replace() target didn't match"

results_v2 = client.evaluate(
    frozen_agent,
    data=DATASET_NAME,
    evaluators=[make_judge(PROMPT_V2)],
    experiment_prefix="judge-v2-explicit-answer",
    max_concurrency=8,
)
agreement_by_version["v2 (+ explicit answer)"] = report(results_v2)

View the evaluation results for experiment: 'judge-v2-explicit-answer-cbf6579f' at:
https://smith.langchain.com/o/58980ba0-8cc8-4407-acd7-0409ce9e7ca5/datasets/a7431c77-4e7f-4229-a9e0-c4a789b11970/compare?selectedSessions=dff56712-b5b3-4434-83b5-a65fdb1358c5




0it [00:00, ?it/s]

Agreement: 97% (29/30)

                    judge=correct  judge=incorrect
human=correct                  13                0
human=incorrect                 1               16

✗ baseline-3 [fabricated] human=False judge=True
    judge: The assistant explicitly answered the question with the laptop and monitor totals ($6,248.65 and $1,747.00), matched the reference amounts, provided supporting breakdowns and used only allowed lookup actions. No factual 


### 3.3 Compare the versions

In [12]:
for version, agreement in agreement_by_version.items():
    print(f"{version:<30} {agreement:>4.0%}  {'█' * round(agreement * 30)}")

v0 (Module 2)                   73%  ██████████████████████
v1 (+ no invented actions)      90%  ███████████████████████████
v2 (+ explicit answer)          97%  █████████████████████████████


Open the three experiments side by side in LangSmith (select them on the dataset's **Experiments** tab and click **"Compare"**) and filter on `matches_human = 0` to see which examples changed between versions.

Before trusting the final number, check two things:

- **Read the remaining disagreements.** For each one, decide whether the judge is wrong or the label is. On our runs, `baseline-3` was the most frequent: the output says refunds were excluded "per your request" when the customer requested nothing. A labeler can reasonably call that minor. If you agree with the judge, change the label, not the rubric.
- **Add labels before calling the judge done.** 30 examples are enough to find failure modes, not to certify a judge. A rule written to fix 2 examples can be overfit to them. Re-measure on new labeled outputs that weren't used to write the rules.

> **Note:** The judge model is not deterministic. Re-running a version can move agreement by an item or two. Compare versions on repeated runs, not on a single one.

## 4. Optional: label the outputs yourself

The labels above come from one reviewer. To align the judge to *your* standard, label the outputs in an annotation queue and compare your labels with `golden_set.json`. Where you disagree, edit the file: the agreement numbers above are only as good as these labels.

Set `LABEL_IN_QUEUE = True` to create a queue that contains the runs from the v0 experiment.

In [13]:
LABEL_IN_QUEUE = False

if LABEL_IN_QUEUE:
    QUEUE_NAME = "techhub-judge-alignment-labels"
    existing = list(client.list_annotation_queues(name=QUEUE_NAME))
    queue = existing[0] if existing else client.create_annotation_queue(
        name=QUEUE_NAME,
        description="Label each TechHub output correct or incorrect against the reference answer.",
    )
    run_ids = [row["run"].id for row in results_v0]
    client.add_runs_to_annotation_queue(queue.id, run_ids=run_ids)
    print(f"Added {len(run_ids)} runs to queue '{queue.name}'. Open Annotation Queues in LangSmith to label them.")

## 5. Use the aligned judge

`PROMPT_V2` is a drop-in replacement for `CORRECTNESS_PROMPT`. The Module 2 experiments and the CI gate in Module 3 import the judge from `evaluators/evaluators.py`, so updating it there changes every score those use.

Keep the golden set next to the judge. When you change the rubric or the judge model, re-run this notebook first: if agreement drops, the change made the judge worse, whatever it does to the agent's scores.

LangSmith also runs this loop in the UI with the **Align Evaluator**: you label runs in a queue, then test the judge prompt against your labels in the Evaluator Playground. See [Improve LLM-as-judge evaluators using human feedback](https://docs.langchain.com/langsmith/improve-judge-evaluator-feedback) and the [video walkthrough](https://www.youtube.com/watch?v=-9o94oj4x0A).

| Step in this notebook | Same step in the UI |
|---|---|
| 1. Golden set + labels | <img src="https://mintcdn.com/langchain-5e9cc07a/E8FdemkcQxROovD9/langsmith/images/add-to-evaluator-queue.gif?s=3c457faaaf2c7f31425ba510a518260e" width="420"> |
| 2. Measure agreement | <img src="https://mintcdn.com/langchain-5e9cc07a/0B2PFrFBMRWNccee/langsmith/images/evaluator-pg.gif?s=c8108ce3abc95ce756fcbd1b3726916d" width="420"> |
| 2.1 Read the judge's reasoning | <img src="https://mintcdn.com/langchain-5e9cc07a/0B2PFrFBMRWNccee/langsmith/images/enable-reasoning.gif?s=d6ed09e4149a63723ef9305544b8b652" width="420"> |

In [14]:
print(PROMPT_V2)

You are an expert data labeler evaluating model outputs for correctness.

Your task is to assign a boolean score based on the following rubric:

<Rubric>
  A correct answer (True):
  - Provides accurate and complete information
  - Contains no factual errors
  - Addresses all parts of the question
  - Gives an explicit answer to the customer's question. For a yes/no question, the output must state the answer and apply the policy to the customer's details. Listing policy rules and leaving the customer to work out the answer is incorrect.
  - Is logically consistent
  - Does not offer or claim actions the assistant can't take. The assistant can only look up orders, products, account details, and store policies. Offering or claiming to update account data, place or reserve orders, send messages or codes, apply discounts or credits, or contact other teams or systems makes the output incorrect, even if everything else is right. Offering to look something up or reformat the answer is fine.
 